# Training Models on Nonlinear Diffusion Dataset

Train and compare three models:
1. **FNO** (Fourier Neural Operator)
2. **U-Net** (Standard encoder-decoder)
3. **MRBNO** (Multilayer Reduced Basis Neural Operator)

on the nonlinear PDE: $-\nabla \cdot (e^{m(x)} \nabla u) = f$


## 1. Load Dataset and Imports


In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import sys
import os

# Add models directory to path
sys.path.append('models')

from FNO_model import FNO2d
from Unet_model import UNet
from MRBNO_model_multichannel import PCAReducerMultiChannel, MultiLayerReducedBasisResNetMultiChannel

print("Libraries imported successfully!")
print("IMPORTANT: If you get UNET errors, restart the kernel (Kernel -> Restart) to reload the fixed model!")


Libraries imported successfully!
IMPORTANT: If you get UNET errors, restart the kernel (Kernel -> Restart) to reload the fixed model!


In [6]:
# Load dataset
data = np.load('data/nonlinear_diffusion_dataset.npz')
f_all = data['f']  # (1000, 100, 100)
m_all = data['m']  # (1000, 100, 100)
u_all = data['u']  # (1000, 100, 100)
N = int(data['N'])
L = float(data['L'])

num_samples = f_all.shape[0]
print(f"Loaded {num_samples} samples")
print(f"Grid size: {N}x{N}")
print(f"Domain: [0, {L}]^2")

# Stack f and m as 2-channel input
X = np.stack([f_all, m_all], axis=1)  # (1000, 2, 100, 100)
Y = u_all  # (1000, 100, 100)

print(f"\nInput X shape: {X.shape}")
print(f"Output Y shape: {Y.shape}")


ValueError: This file contains pickled (object) data. If you trust the file you can load it unsafely using the `allow_pickle=` keyword argument or `pickle.load()`.

## 2. Normalize and Split Data


In [3]:
# Normalize
x_mean = X.mean()
x_std = X.std()
y_mean = Y.mean()
y_std = Y.std()

Xn = (X - x_mean) / (x_std + 1e-8)
Yn = (Y - y_mean) / (y_std + 1e-8)

print(f"X normalization: mean={x_mean:.4f}, std={x_std:.4f}")
print(f"Y normalization: mean={y_mean:.4f}, std={y_std:.4f}")

# Train/val split
train_frac = 0.7
n_train = int(train_frac * num_samples)

X_train, Y_train = Xn[:n_train], Yn[:n_train]
X_val, Y_val = Xn[n_train:], Yn[n_train:]

print(f"\nTraining samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")


X normalization: mean=0.0379, std=1.7312
Y normalization: mean=0.0040, std=0.0888

Training samples: 700
Validation samples: 300


## 3. Prepare Data and PCA


In [4]:
# Convert to PyTorch tensors
X_train_t = torch.from_numpy(X_train).float()  # (700, 2, 100, 100)
Y_train_t = torch.from_numpy(Y_train).float().unsqueeze(1)  # (700, 1, 100, 100)
X_val_t = torch.from_numpy(X_val).float()  # (300, 2, 100, 100)
Y_val_t = torch.from_numpy(Y_val).float().unsqueeze(1)  # (300, 1, 100, 100)

print(f"Training data: X={X_train_t.shape}, Y={Y_train_t.shape}")
print(f"Validation data: X={X_val_t.shape}, Y={Y_val_t.shape}")

# Create data loaders
batch_size = 16
train_loader = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, Y_val_t), batch_size=batch_size, shuffle=False)

# Compute PCA for MRBNO
rb = PCAReducerMultiChannel().fit(Xn, Yn)
epsilon = 1e-5
k_in, k_out = rb.k_from_cutoff(j_in=1-1*epsilon, j_out=1-epsilon)

print(f"\nPCA components: k_in={k_in}, k_out={k_out}")


Training data: X=torch.Size([700, 2, 100, 100]), Y=torch.Size([700, 1, 100, 100])
Validation data: X=torch.Size([300, 2, 100, 100]), Y=torch.Size([300, 1, 100, 100])

PCA components: k_in=122, k_out=61


## 4. Training Function


In [5]:
def train_model(model, train_loader, val_loader, epochs=100, lr=1e-3, device='cpu'):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-6)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.8)
    criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                loss = criterion(pred, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_loader.dataset)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
        
        scheduler.step()
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"[{epoch+1:03d}/{epochs}] train={train_loss:.4e} val={val_loss:.4e} lr={scheduler.get_last_lr()[0]:.2e} (best: {best_val_loss:.4e})")
    
    return train_losses, val_losses, best_val_loss


## 5. Train All Three Models


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else 
                      "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


## 5a. Train FNO Model


In [7]:
# Train FNO
print("\n" + "="*70)
print("TRAINING FNO")
print("="*70)
fno_model = FNO2d(in_channels=2, out_channels=1, width=32, modes=(16, 16), n_layers=4)
print(f"Parameters: {sum(p.numel() for p in fno_model.parameters()):,}")
fno_train_losses, fno_val_losses, fno_best_val = train_model(
    fno_model, train_loader, val_loader, epochs=150, lr=5e-3, device=device
)

# Save model
torch.save(fno_model.state_dict(), 'checkpoints/fno_nonlinear_best.pt')
print("FNO model saved to checkpoints/fno_nonlinear_best.pt")



TRAINING FNO
Parameters: 1,054,241


KeyboardInterrupt: 

## 5b. Train U-Net Model


In [ ]:
# Train U-Net
print("\n" + "="*70)
print("TRAINING U-NET")
print("="*70)
unet_model = UNet(in_channels=2, out_channels=1, base_features=32, depth=3, bilinear=True)
print(f"Parameters: {sum(p.numel() for p in unet_model.parameters()):,}")
unet_train_losses, unet_val_losses, unet_best_val = train_model(
    unet_model, train_loader, val_loader, epochs=150, lr=5e-3, device=device
)

# Save model
torch.save(unet_model.state_dict(), 'checkpoints/unet_nonlinear_best.pt')
print("U-Net model saved to checkpoints/unet_nonlinear_best.pt")



TRAINING U-NET
Parameters: 1,949,409
[001/150] train=6.0083e-01 val=7.7769e+00 lr=5.00e-03 (best: 7.7769e+00)
[010/150] train=3.4428e-01 val=3.1632e-01 lr=5.00e-03 (best: 8.1483e-02)


KeyboardInterrupt: 

## 5c. Train MRBNO Model


In [ ]:
# Train MRBNO
print("\n" + "="*70)
print("TRAINING MRBNO")
print("="*70)
mrbno_model = MultiLayerReducedBasisResNetMultiChannel(
    rb=rb, k_in=k_in, k_out=k_out, layers=5, hidden=64, depth=2
)
print(f"Parameters: {sum(p.numel() for p in mrbno_model.parameters()):,}")
mrbno_train_losses, mrbno_val_losses, mrbno_best_val = train_model(
    mrbno_model, train_loader, val_loader, epochs=150, lr=5e-3, device=device
)

# Save model
torch.save(mrbno_model.state_dict(), 'checkpoints/mrbno_nonlinear_best.pt')
print("MRBNO model saved to checkpoints/mrbno_nonlinear_best.pt")

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)



TRAINING MRBNO
Parameters: 152,884
[001/150] train=9.7818e-01 val=5.4549e-01 lr=5.00e-03 (best: 5.4549e-01)


## 6. Compare Results


In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fno_train_losses, label='FNO', linewidth=2)
axes[0].plot(unet_train_losses, label='U-Net', linewidth=2)
axes[0].plot(mrbno_train_losses, label='MRBNO', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss (MSE)')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

axes[1].plot(fno_val_losses, label=f'FNO (best: {fno_best_val:.4e})', linewidth=2)
axes[1].plot(unet_val_losses, label=f'U-Net (best: {unet_best_val:.4e})', linewidth=2)
axes[1].plot(mrbno_val_losses, label=f'MRBNO (best: {mrbno_best_val:.4e})', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Loss (MSE)')
axes[1].set_title('Validation Loss Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

# Print summary
print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)
print(f"\nFNO:   Best Val Loss = {fno_best_val:.6e}, RMSE = {np.sqrt(fno_best_val):.6e}")
print(f"U-Net: Best Val Loss = {unet_best_val:.6e}, RMSE = {np.sqrt(unet_best_val):.6e}")
print(f"MRBNO: Best Val Loss = {mrbno_best_val:.6e}, RMSE = {np.sqrt(mrbno_best_val):.6e}")


## 7. Visualize Predictions


In [ ]:
# Pick validation sample and get predictions
idx = 5
x_sample = X_val_t[idx:idx+1].to(device)
y_true = Y_val_t[idx].cpu().numpy()[0]

fno_model.eval()
unet_model.eval()
mrbno_model.eval()

with torch.no_grad():
    fno_pred = fno_model(x_sample).cpu().numpy()[0, 0]
    unet_pred = unet_model(x_sample).cpu().numpy()[0, 0]
    mrbno_pred = mrbno_model(x_sample).cpu().numpy()[0, 0]

# Denormalize
f_sample = (x_sample[0, 0].cpu().numpy() * x_std + x_mean)
m_sample = (x_sample[0, 1].cpu().numpy() * x_std + x_mean)
y_true_denorm = y_true * y_std + y_mean
fno_pred_denorm = fno_pred * y_std + y_mean
unet_pred_denorm = unet_pred * y_std + y_mean
mrbno_pred_denorm = mrbno_pred * y_std + y_mean

# Plot
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

# Row 1: Inputs
axes[0, 0].imshow(f_sample, origin='lower', cmap='RdBu_r')
axes[0, 0].set_title('Input: f(x,y)')
axes[0, 1].imshow(m_sample, origin='lower', cmap='viridis')
axes[0, 1].set_title('Input: m(x,y)')
axes[0, 2].imshow(np.exp(m_sample), origin='lower', cmap='plasma')
axes[0, 2].set_title('D = exp(m)')
axes[0, 3].imshow(y_true_denorm, origin='lower', cmap='coolwarm')
axes[0, 3].set_title('Ground Truth u')

# Row 2: Predictions
vmin, vmax = y_true_denorm.min(), y_true_denorm.max()
axes[1, 0].imshow(fno_pred_denorm, origin='lower', cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[1, 0].set_title('FNO Prediction')
axes[1, 1].imshow(unet_pred_denorm, origin='lower', cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[1, 1].set_title('U-Net Prediction')
axes[1, 2].imshow(mrbno_pred_denorm, origin='lower', cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[1, 2].set_title('MRBNO Prediction')
axes[1, 3].axis('off')

# Row 3: Errors
fno_err = np.abs(fno_pred_denorm - y_true_denorm)
unet_err = np.abs(unet_pred_denorm - y_true_denorm)
mrbno_err = np.abs(mrbno_pred_denorm - y_true_denorm)
err_max = max(fno_err.max(), unet_err.max(), mrbno_err.max())

axes[2, 0].imshow(fno_err, origin='lower', cmap='magma', vmin=0, vmax=err_max)
axes[2, 0].set_title(f'FNO Error (MSE: {np.mean(fno_err**2):.4e})')
axes[2, 1].imshow(unet_err, origin='lower', cmap='magma', vmin=0, vmax=err_max)
axes[2, 1].set_title(f'U-Net Error (MSE: {np.mean(unet_err**2):.4e})')
axes[2, 2].imshow(mrbno_err, origin='lower', cmap='magma', vmin=0, vmax=err_max)
axes[2, 2].set_title(f'MRBNO Error (MSE: {np.mean(mrbno_err**2):.4e})')
axes[2, 3].axis('off')

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(f'Model Comparison: -∇·(exp(m)∇u)=f (Sample #{idx})', fontsize=16)
plt.tight_layout()
plt.show()
